### Подключение модулей

In [1]:
import torch
import numpy as np
import random
import tabulate
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType, PrefixTuningConfig
from sklearn.metrics import accuracy_score, f1_score
import time
import pandas as pd

### Подготовка

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 16
EPOCHS = 3
MAX_LENGTH = 128
NUM_LABELS = 6

In [3]:
model_names = [
    "full_finetuning",
    "linear_probing",
    "prefix_tuning",
    "lora_tuning"
]

#### Загрузка датасета

In [4]:
dataset = load_dataset("dair-ai/emotion")

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

In [7]:
def compute_metrics(p):
    labels = p.label_ids
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc,
            "f1": f1}

In [8]:
def train_model(model, train_data, eval_data, name, peft_cfg=None):
    if peft_cfg:
        model = get_peft_model(model, peft_cfg)

    training_args = TrainingArguments(
        output_dir=f"./models/{name}",
        eval_strategy="epoch",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        seed=SEED,
        save_strategy="no",
        logging_dir=f"./logs/{name}",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics
    )

    print("Метрики ДО дообучения:")
    pre_training_results = trainer.evaluate()
    print(tabulate.tabulate(
        pre_training_results.items(),
        headers=["Метрика", "Значение"],
        tablefmt="grid",
        floatfmt=".4f"
    ))

    start = time.time()
    trainer.train()
    end = time.time()

    print("Метрики ПОСЛЕ дообучения:")
    post_training_results = trainer.evaluate()
    print(tabulate.tabulate(
        post_training_results.items(),
        headers=["Метрика", "Значение"],
        tablefmt="grid",
        floatfmt=".4f"
    ))

    post_training_results.update({
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad), # Количество обучаемых параметров
        "total_params": sum(p.numel() for p in model.parameters()),                        # Общее количество параметров
        "training_time_min": (end - start) / 60,                                           # Время обучения
        "max_gpu_memory_mb": torch.cuda.max_memory_allocated() / 1024**2                   # Количество выделенной памяти
    })
    return post_training_results

In [9]:
tokenized_datasets = dataset.map(preprocess)

In [10]:
id2label = dataset["train"].features["label"].int2str
label2id = dataset["train"].features["label"].str2int

### Обучение модели в режиме full finetuning

Обучаются все параметры модели

In [11]:
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
results_full = train_model(model_full,
                           tokenized_datasets["train"],
                           tokenized_datasets["validation"],
                           name=model_names[0])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Метрики ДО дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     1.6927 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0021 |
+-----------------------------+------------+
| eval_accuracy               |     0.1430 |
+-----------------------------+------------+
| eval_f1                     |     0.0678 |
+-----------------------------+------------+
| eval_runtime                |    16.1207 |
+-----------------------------+------------+
| eval_samples_per_second     |   124.0640 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.7540 |
+-----------------------------+------------+


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,0.221600,0.185019,0.002100,0.929000,0.903944
2,0.125900,0.149893,0.002100,0.935000,0.907823
3,0.083600,0.175635,0.002100,0.937000,0.909486


Метрики ПОСЛЕ дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     0.1756 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0021 |
+-----------------------------+------------+
| eval_accuracy               |     0.9370 |
+-----------------------------+------------+
| eval_f1                     |     0.9095 |
+-----------------------------+------------+
| eval_runtime                |    16.9308 |
+-----------------------------+------------+
| eval_samples_per_second     |   118.1280 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.3830 |
+-----------------------------+------------+
| epoch                       |     3.0000 |
+-----------------------------+------------+


### Обучение модели в режиме linear probing

`Linear` → `ReLU` → `Dropout` → `Linear` — это типичная архитектура небольшой feedforward-сети, которая:

- Позволяет лучше захватывать сложные зависимости в представлениях
- Повышает обучаемость и экспрессивность модели
- `Dropout` служит для регуляризации, чтобы избежать переобучения


Обучается новая классификационная голова, написанная ниже

In [12]:
class LinearProbeHead(torch.nn.Module):
    def __init__(self, hidden_size, num_labels):
        super().__init__()
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_size, num_labels)
        )

    def forward(self, x):
        return self.classifier(x)

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
for param in base_model.bert.parameters():
    param.requires_grad = False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
results_linear = train_model(base_model,
                             tokenized_datasets["train"],
                             tokenized_datasets["validation"],
                             name=model_names[1])

Метрики ДО дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     1.9527 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0020 |
+-----------------------------+------------+
| eval_accuracy               |     0.1160 |
+-----------------------------+------------+
| eval_f1                     |     0.0777 |
+-----------------------------+------------+
| eval_runtime                |    16.7653 |
+-----------------------------+------------+
| eval_samples_per_second     |   119.2940 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.4560 |
+-----------------------------+------------+


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,1.574600,1.565783,0.002000,0.385000,0.139558
2,1.558400,1.555619,0.002000,0.386500,0.147394
3,1.558300,1.553160,0.002000,0.397000,0.151997


Метрики ПОСЛЕ дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     1.5532 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0020 |
+-----------------------------+------------+
| eval_accuracy               |     0.3970 |
+-----------------------------+------------+
| eval_f1                     |     0.1520 |
+-----------------------------+------------+
| eval_runtime                |    17.0384 |
+-----------------------------+------------+
| eval_samples_per_second     |   117.3820 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.3360 |
+-----------------------------+------------+
| epoch                       |     3.0000 |
+-----------------------------+------------+


### Обучение модели в режиме PEFT с использованием prompt tuning или prefix tuning.


1. По точности `Prefix Tuning` может уступать `full finetuning` для сложных задач, но догоняет на больших моделях. При этом обновляет меньше параметров (только виртуальные токены), поэтому вычислительные затраты меньше.
2. В отличие от `prompt tuning` (который более эффективен для больших моделей и генерации), `prefix tuning` отлично работает в задачах `sequence classification` (`emotion` — классификация эмоций по тексту).

In [14]:
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,
    num_layers=12,
    encoder_hidden_size=768
)

model_prefix = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
results_prefix = train_model(model_prefix,
                             tokenized_datasets["train"],
                             tokenized_datasets["validation"],
                             name=model_names[2],
                             peft_cfg=prefix_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Метрики ДО дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     1.9275 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0020 |
+-----------------------------+------------+
| eval_accuracy               |     0.1225 |
+-----------------------------+------------+
| eval_f1                     |     0.0472 |
+-----------------------------+------------+
| eval_runtime                |    17.7606 |
+-----------------------------+------------+
| eval_samples_per_second     |   112.6090 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.0380 |
+-----------------------------+------------+


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,1.872900,1.841951,0.002000,0.174000,0.103341
2,1.782500,1.742275,0.002000,0.260500,0.132881
3,1.745300,1.720073,0.002000,0.285000,0.139025


Метрики ПОСЛЕ дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     1.7201 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0020 |
+-----------------------------+------------+
| eval_accuracy               |     0.2850 |
+-----------------------------+------------+
| eval_f1                     |     0.1390 |
+-----------------------------+------------+
| eval_runtime                |    17.7501 |
+-----------------------------+------------+
| eval_samples_per_second     |   112.6750 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.0420 |
+-----------------------------+------------+
| epoch                       |     3.0000 |
+-----------------------------+------------+


### Обучение модели в режиме PEFT с использованием LoRA.

* `lora_alpha`: Значение 32 дает стабильное обучение, лучше, чем 16/64
* `r`: Значение 8 дает хорошее качество при минимальной нагрузке по памяти и числу параметров. При `r`=16 качество росло незначительно, а ресурсы расходовались почти вдвое больше.

In [15]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model_lora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
results_lora = train_model(model_lora,
                           tokenized_datasets["train"],
                           tokenized_datasets["validation"],
                           name=model_names[3],
                           peft_cfg=lora_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Метрики ДО дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     1.9527 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0045 |
+-----------------------------+------------+
| eval_accuracy               |     0.1160 |
+-----------------------------+------------+
| eval_f1                     |     0.0777 |
+-----------------------------+------------+
| eval_runtime                |    17.7381 |
+-----------------------------+------------+
| eval_samples_per_second     |   112.7520 |
+-----------------------------+------------+
| eval_steps_per_second       |     7.0470 |
+-----------------------------+------------+


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,1.134700,0.978102,0.004500,0.627500,0.369955
2,0.870300,0.799390,0.004500,0.716000,0.539586
3,0.774300,0.737849,0.004500,0.729000,0.567377


Метрики ПОСЛЕ дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     0.7378 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0045 |
+-----------------------------+------------+
| eval_accuracy               |     0.7290 |
+-----------------------------+------------+
| eval_f1                     |     0.5674 |
+-----------------------------+------------+
| eval_runtime                |    17.8983 |
+-----------------------------+------------+
| eval_samples_per_second     |   111.7420 |
+-----------------------------+------------+
| eval_steps_per_second       |     6.9840 |
+-----------------------------+------------+
| epoch                       |     3.0000 |
+-----------------------------+------------+


`r` - размерность low-rank матриц \
`lora_alpha` - масштаб параметров LoRA \
`lora_dropout` - Dropout на LoRA-слоях

### Выводим метрики

In [16]:
results_df = pd.DataFrame([
    {"method": "Full Finetuning", **results_full},
    {"method": "Linear Probing", **results_linear},
    {"method": "Prefix Tuning", **results_prefix},
    {"method": "LoRA (r=8)", **results_lora},
])

result_table = tabulate.tabulate(
    results_df,
    headers='keys',
    tablefmt='grid',
    showindex=False,
    floatfmt=".4f"
)

print(result_table)

results_df.to_csv("results.csv", index=False)

+-----------------+-------------+-------------------------------+-----------------+-----------+----------------+---------------------------+-------------------------+---------+--------------------+----------------+---------------------+---------------------+
| method          |   eval_loss |   eval_model_preparation_time |   eval_accuracy |   eval_f1 |   eval_runtime |   eval_samples_per_second |   eval_steps_per_second |   epoch |   trainable_params |   total_params |   training_time_min |   max_gpu_memory_mb |
+=================+=============+===============================+=================+===========+================+===========================+=========================+=========+====================+================+=====================+=====================+
| Full Finetuning |      0.1756 |                        0.0021 |          0.9370 |    0.9095 |        16.9308 |                  118.1280 |                  7.3830 |  3.0000 |          109486854 |      109486854 |         

### Выводы:

Видно, что `Full Finetuning` показывает лучшие результаты по всем метрикам качества.

Разница во времени обучения есть. Например, `Linear Probing` почти в 3 раза быстрее обучается, чем `Full Finetuning` (`training_time_min`).
Остальные модели оказались почти в полтора раза быстрее в обучении, при этом `LoRA` показала результаты лучше, чем `Linear Probing` и `Prefix Tuning`.

`LoRA` — единственный метод PEFT, показывающий неплохую точность.

`Prefix` и `Linear probing` работают очень слабо — вероятно, классификационная задача слишком сложна для адаптации через небольшое число параметров.
